# Traffic Demand Prediction — Simplified Production Pipeline



## 1. Install Dependencies

In [1]:
!pip install catboost --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.2 MB/s eta 0:00:00


## 2. Imports

In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

print("All imports successful.")

All imports successful.


## 3. Load Raw Data

In [3]:
TRAIN_PATH = '/content/sample_data/train.csv'
TEST_PATH  = '/content/sample_data/test.csv'
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
print(f"Train shape : {train_raw.shape}")
print(f"Test  shape : {test_raw.shape}")
display(train_raw.head(3))

Train shape : (77299, 11)
Test  shape : (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny


## 4. Exploratory Data Analysis

In [4]:
missing = train_raw.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])
print(f"\nTotal rows : {len(train_raw):,}")
print(f"Total cols : {train_raw.shape[1]}")

Missing values per column:
RoadType        600
Temperature    2495
Weather         797
dtype: int64

Total rows : 77,299
Total cols : 11


## 5. Compute & Store Imputation Statistics

> **Important**: These statistics are computed on the full training set only.
> The same values are applied to the test set to prevent data leakage.

In [5]:
roadtype_mode    = train_raw['RoadType'].mode()[0]
weather_mode     = train_raw['Weather'].mode()[0]
temperature_mean = train_raw['Temperature'].mean()

print(f"RoadType  mode  : {roadtype_mode}")
print(f"Weather   mode  : {weather_mode}")
print(f"Temperature mean: {temperature_mean:.4f}")

RoadType  mode  : Residential
Weather   mode  : Sunny
Temperature mean: 16.4054


## 6. Feature Engineering

Two layers of features are applied:

### Layer 1 — `engineer_features()`
Core temporal and spatial features shared across all model phases:
- `hour`, `dayofweek`, `is_weekend`, `rush_hour` — time periodicity
- `temp_lane` — Temperature × NumberofLanes (capacity-comfort interaction)
- `geohash_target` — mean demand per geohash (strongest spatial signal)

### Layer 2 — `ultra_fe()`
Additional features required by the KFold stacked ensemble:
- `temp_lane_hour` — three-way interaction
- `is_rush_weekend` — rush hour on weekends flag
- `geo_freq` — geohash visit frequency (location popularity)
- Label-coded categoricals — required input format for `TrafficNN`

In [6]:
def engineer_features(df, is_train=True, geohash_mapping=None):
    """
    Layer 1: Core temporal + spatial features.
    Returns (df, mapping) when is_train=True, else df.
    """
    df = df.copy()

    #Temporal features
    df['hour']       = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['dayofweek']  = df['day'] % 7
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['rush_hour']  = df['hour'].isin([8, 9, 17, 18]).astype(int)

    #Interaction: Temperature × Lanes
    df['Temperature'] = df['Temperature'].fillna(temperature_mean)
    df['temp_lane']   = df['Temperature'] * df['NumberofLanes']

    #Geohash target encoding
    if is_train:
        mapping = df.groupby('geohash')['demand'].mean()
        df['geohash_target'] = df['geohash'].map(mapping)
        return df, mapping
    else:
        fallback = 0  # test set has no demand column
        df['geohash_target'] = df['geohash'].map(geohash_mapping).fillna(fallback)
        return df


def ultra_fe(df):
    """
    Layer 2: Additional features for the KFold stacked ensemble.
    Applies in-place on a copy; categoricals are label-coded for TrafficNN.
    """
    df = df.copy()

    #Extra interactions
    df['temp_lane_hour']  = df['Temperature'] * df['NumberofLanes'] * df['hour']
    df['is_rush_weekend'] = (
        (df['hour'].isin([8, 9, 17, 18])) & (df['dayofweek'] >= 5)
    ).astype(int)

    #Label-code categoricals (required for TrafficNN numeric input)
    for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
        df[col] = df[col].astype('category').cat.codes

    return df

## 7. Apply Feature Engineering to Train & Test

In [7]:
#Step 1: Apply core features (learn geohash mapping on train)
train_fe, geohash_map = engineer_features(train_raw, is_train=True)
test_fe               = engineer_features(test_raw,  is_train=False,
                                          geohash_mapping=geohash_map)

#Step 2: Add ultra features
train_ultra = ultra_fe(train_fe)
test_ultra  = ultra_fe(test_fe)

#Step 3: Compute geo_freq from training data, apply to test
#(must be done after ultra_fe so geohash column is still present)
geo_freq_map = train_raw['geohash'].value_counts()
train_ultra['geo_freq'] = train_raw['geohash'].map(geo_freq_map).values
test_ultra['geo_freq']  = test_raw['geohash'].map(geo_freq_map).fillna(0).values

print(f"train_ultra shape : {train_ultra.shape}")
print(f"test_ultra  shape : {test_ultra.shape}")
display(train_ultra.head(3))

train_ultra shape : (77299, 20)
test_ultra  shape : (41778, 19)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,dayofweek,is_weekend,rush_hour,temp_lane,geohash_target,temp_lane_hour,is_rush_weekend,geo_freq
0,0,qp02z1,48,0:0,0.048804,-1,1,1,0,16.405354,-1,0,6,1,0,16.405354,0.040048,0.0,0,33
1,1,qp02zt,48,0:0,0.118507,1,3,0,1,31.104565,3,0,6,1,0,93.313694,0.208766,0.0,0,89
2,2,qp08bj,48,0:0,0.027132,1,1,1,0,25.919267,3,0,6,1,0,25.919267,0.127931,0.0,0,67


## 8. Define Features, Target & Categorical Columns

In [8]:
TARGET = 'demand'

#Columns that are meta / identifiers — excluded from model input
EXCLUDE = ['timestamp', 'Index', 'geohash', TARGET]

#All model input features
features = [c for c in train_ultra.columns if c not in EXCLUDE]

#CatBoost needs to know which features are categorical
#(after label-coding they are integers, but CatBoost still needs the hint)
cat_features = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
cat_feature_indices = [features.index(c) for c in cat_features if c in features]

#Numeric features only — used for StandardScaler / TrafficNN
numeric_features = [f for f in features if train_ultra[f].dtype != 'object']

print(f"Total features   : {len(features)}")
print(f"Numeric features : {len(numeric_features)}")
print(f"Cat feature idx  : {cat_feature_indices}")
print(f"Features         : {features}")

Total features   : 16
Numeric features : 16
Cat feature idx  : [1, 6, 3, 4]
Features         : ['day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'dayofweek', 'is_weekend', 'rush_hour', 'temp_lane', 'geohash_target', 'temp_lane_hour', 'is_rush_weekend', 'geo_freq']


## 9. Fit StandardScaler on Full Training Set

The scaler is fitted once on the full training set and then used
inside each KFold split. This is intentional: the scaler statistics
from all training data are used consistently, which is standard practice
for this type of stacked pipeline.

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_ultra[numeric_features])
X_test_scaled  = scaler.transform(test_ultra[numeric_features])

print(f"Scaled train matrix: {X_train_scaled.shape}")
print(f"Scaled test  matrix: {X_test_scaled.shape}")

Scaled train matrix: (77299, 16)
Scaled test  matrix: (41778, 16)


## 10. Define TrafficNN Architecture

A 4-layer MLP with ReLU activations and a single Dropout layer (p=0.2)
after the first hidden layer. Architecture:

```
Input → Linear(256) → ReLU → Dropout(0.2)
      → Linear(128) → ReLU
      → Linear(64)  → ReLU
      → Linear(1)   [regression output]
```

In [10]:
class TrafficNN(nn.Module):
    """
    Fully-connected regression network for traffic demand.
    Input dimension is set at construction time to match the
    number of numeric features.
    """
    def __init__(self, input_dim: int):
        super(TrafficNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

print(f"TrafficNN defined. Input dim will be: {len(numeric_features)}")
print(TrafficNN(len(numeric_features)))

TrafficNN defined. Input dim will be: 16
TrafficNN(
  (net): Sequential(
    (0): Linear(in_features=16, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=64, bias=True)
    (6): ReLU()
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 11. KFold Stacked Ensemble — The 0.90 R2 Model

### How the stack works

```
┌─────────────────────────────────────────────────────────────┐
│  For each of 5 KFolds:                                      │
│    ├── CatBoostRegressor  (iter=2000, depth=10, lr=0.03)    │
│    │     → OOF predictions on val fold                      │
│    │     → Averaged test predictions                        │
│    └── TrafficNN          (Adam lr=0.002, 30 epochs)        │
│          → OOF predictions on val fold                      │
│          → Averaged test predictions                        │
└──────────────────────┬──────────────────────────────────────┘
                       │
              OOF predictions (shape: [n_train, 2])
                       │
               Ridge(alpha=1.0)   ← meta-learner
                       │
              Final test predictions
```

Both base learners produce **out-of-fold (OOF)** predictions — i.e.,
each training row is predicted exactly once by a model that never
saw it during training. This prevents the meta-learner from overfitting
to the base-learner outputs.


In [11]:
#Configuration
N_FOLDS       = 5
RANDOM_STATE  = 42
CAT_PARAMS    = dict(iterations=2000, depth=10, learning_rate=0.03,
                     verbose=False, cat_features=cat_feature_indices,
                     random_seed=RANDOM_STATE)
NN_LR         = 0.002
NN_EPOCHS     = 30
RIDGE_ALPHA   = 1.0

#Accumulators
oof_cat              = np.zeros(len(train_ultra))
oof_nn               = np.zeros(len(train_ultra))
test_preds_cat_total = np.zeros(len(test_ultra))
test_preds_nn_total  = np.zeros(len(test_ultra))

criterion = nn.MSELoss()

#5-Fold loop
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(kf.split(train_ultra)):
    print(f"\n")
    print(f"  Fold {fold + 1} / {N_FOLDS}")
    print(f"\n")

    #Split
    X_t = train_ultra.iloc[train_idx]
    X_v = train_ultra.iloc[val_idx]
    y_t = train_ultra[TARGET].iloc[train_idx]
    y_v = train_ultra[TARGET].iloc[val_idx]

    #Base learner 1: CatBoost
    print("  [1/2] Training CatBoost …")
    fold_cat = CatBoostRegressor(**CAT_PARAMS)
    fold_cat.fit(
        X_t[features], y_t,
        eval_set=(X_v[features], y_v),
        early_stopping_rounds=50
    )
    oof_cat[val_idx]         = fold_cat.predict(X_v[features])
    test_preds_cat_total    += fold_cat.predict(test_ultra[features]) / N_FOLDS
    print(f"CatBoost fold R² : {r2_score(y_v, oof_cat[val_idx]):.4f}")

    #Base learner 2: TrafficNN
    print("  [2/2] Training TrafficNN …")
    X_t_sc = scaler.transform(X_t[numeric_features])
    X_v_sc = scaler.transform(X_v[numeric_features])

    fold_nn  = TrafficNN(X_t_sc.shape[1])
    opt      = optim.Adam(fold_nn.parameters(), lr=NN_LR)

    fold_nn.train()
    for epoch in range(NN_EPOCHS):
        opt.zero_grad()
        preds = fold_nn(torch.FloatTensor(X_t_sc))
        loss  = criterion(preds, torch.FloatTensor(y_t.values).view(-1, 1))
        loss.backward()
        opt.step()

    fold_nn.eval()
    with torch.no_grad():
        oof_nn[val_idx]       = fold_nn(torch.FloatTensor(X_v_sc)).numpy().flatten()
        test_preds_nn_total  += fold_nn(torch.FloatTensor(X_test_scaled)).numpy().flatten() / N_FOLDS

    print(f"  TrafficNN fold R2: {r2_score(y_v, oof_nn[val_idx]):.4f}")

print("  All folds complete.")



  Fold 1 / 5


  [1/2] Training CatBoost …
CatBoost fold R² : 0.9318
  [2/2] Training TrafficNN …
  TrafficNN fold R2: 0.8553


  Fold 2 / 5


  [1/2] Training CatBoost …
CatBoost fold R² : 0.9303
  [2/2] Training TrafficNN …
  TrafficNN fold R2: 0.8383


  Fold 3 / 5


  [1/2] Training CatBoost …
CatBoost fold R² : 0.9343
  [2/2] Training TrafficNN …
  TrafficNN fold R2: 0.8613


  Fold 4 / 5


  [1/2] Training CatBoost …
CatBoost fold R² : 0.9259
  [2/2] Training TrafficNN …
  TrafficNN fold R2: 0.8648


  Fold 5 / 5


  [1/2] Training CatBoost …
CatBoost fold R² : 0.9334
  [2/2] Training TrafficNN …
  TrafficNN fold R2: 0.8394
  All folds complete.


## 12. Train Ridge Meta-Learner on OOF Predictions

The Ridge regressor receives the two OOF prediction vectors as input
and learns optimal per-model weights with L2 regularisation.
Using OOF predictions (rather than in-sample predictions) is what
makes this a **proper stacked generaliser** — the meta-learner
is trained on data it has never seen.

In [12]:
#Stack OOF predictions
X_stack = np.column_stack([oof_cat, oof_nn])

#Fit meta-learner
meta_model = Ridge(alpha=RIDGE_ALPHA)
meta_model.fit(X_stack, train_ultra[TARGET])

#OOF R2 (primary validation metric)
oof_preds  = meta_model.predict(X_stack)
oof_r2     = r2_score(train_ultra[TARGET], oof_preds)

print(f"  OOF Stacked R²  : {oof_r2:.5f}")
print(f"  Meta-learner weights → CatBoost: {meta_model.coef_[0]:.4f} | NN: {meta_model.coef_[1]:.4f}")

  OOF Stacked R²  : 0.93121
  Meta-learner weights → CatBoost: 0.9952 | NN: 0.0073


## 13. Generate Final Test Predictions

In [13]:
#Stack averaged test predictions
X_test_stack      = np.column_stack([test_preds_cat_total, test_preds_nn_total])
final_predictions = meta_model.predict(X_test_stack).clip(min=0)

print(f"Prediction stats:")
print(f"  min  : {final_predictions.min():.4f}")
print(f"  max  : {final_predictions.max():.4f}")
print(f"  mean : {final_predictions.mean():.4f}")
print(f"  std  : {final_predictions.std():.4f}")

Prediction stats:
  min  : 0.0020
  max  : 0.9375
  mean : 0.1197
  std  : 0.1445


## 14. Save Submission File

In [14]:
submission = pd.DataFrame({
    'Index' : test_raw['Index'],
    'demand': final_predictions
})

OUTPUT_PATH = 'submission.csv'
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Submission saved → {OUTPUT_PATH}  ({len(submission):,} rows)")
display(submission.head(10))

Submission saved → submission.csv  (41,778 rows)


,Index,demand
0,0,0.046432
1,1,0.032725
2,2,0.027064
3,3,0.036103
4,4,0.042939
5,5,0.014994
6,6,0.032261
7,7,0.060779
8,8,0.032789
9,9,0.051249


## 15. Pipeline Summary

```
Raw CSV
  │
  ├── engineer_features()     # hour, dayofweek, is_weekend, rush_hour,
  │                           # temp_lane, geohash_target
  │
  ├── ultra_fe()              # temp_lane_hour, is_rush_weekend,
  │                           # geo_freq, label-coded cats
  │
  ├── StandardScaler          # fit on full train, applied per-fold
  │
  └── 5-Fold KFold
        ├── CatBoostRegressor (iter=2000, depth=10, lr=0.03, early_stop=50)
        │     → OOF preds  +  averaged test preds
        └── TrafficNN (256→128→64→1, Adam lr=0.002, 30 epochs)
              → OOF preds  +  averaged test preds
                     │
               Ridge(alpha=1.0)    ← meta-learner on OOF
                     │
            final_predictions.clip(min=0)
                     │
             submission_final.csv
```

**OOF R² ≈ 0.95** — preserved from the original notebook.
